In [3]:
import subprocess
result = subprocess.run(['pip', 'show', 'azure-ai-ml'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

In [4]:
import subprocess
subprocess.run(['pip', 'install', 'azure-ai-ml', 'azure-identity', '--quiet'])
print("Done")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
adlfs 2025.8.0 requires fsspec>=2023.12.0, but you have fsspec 2023.10.0 which is incompatible.
azure-cli 2.81.0 requires azure-datalake-store~=1.0.1, but you have azure-datalake-store 0.0.53 which is incompatible.
azure-cli 2.81.0 requires azure-keyvault-keys==4.11.0, but you have azure-keyvault-keys 4.8.0 which is incompatible.
azure-cli 2.81.0 requires azure-mgmt-keyvault==12.1.0, but you have azure-mgmt-keyvault 10.3.1 which is incompatible.
azure-cli 2.81.0 requires azure-mgmt-storage==24.0.0, but you have azure-mgmt-storage 22.0.0 which is incompatible.
azure-cli 2.81.0 requires websocket-client~=1.3.1, but you have websocket-client 1.9.0 which is incompatible.
azureml-automl-dnn-nlp 1.61.0 requires torch==2.2.2, but you have torch 2.9.1 which is incompatible.
azureml-automl-runtime 1.61.0 requires psutil<5.

Done


In [1]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import os

ml_client = MLClient.from_config(credential=DefaultAzureCredential())
print("Connected:", ml_client.workspaces.get("ml-learning-workspace").name)

Found the config file in: /config.json


Connected: ml-learning-workspace


In [6]:
# Train model and save it locally
data = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

accuracy = accuracy_score(y_test, model.predict(X_test))
print(f"Accuracy: {accuracy}")

# Save model to disk
import joblib
os.makedirs("model", exist_ok=True)
joblib.dump(model, "model/iris_model.pkl")
print("Model saved to model/iris_model.pkl")

Accuracy: 1.0
Model saved to model/iris_model.pkl


In [7]:
# Register model in Azure ML
model = Model(
    path="model",
    name="iris-random-forest",
    description="Random Forest classifier trained on Iris dataset",
    type=AssetTypes.CUSTOM_MODEL
)

registered_model = ml_client.models.create_or_update(model)
print(f"Model registered: {registered_model.name}, version: {registered_model.version}")

Uploading model (0.19 MBs): 100%|██████████| 187265/187265 [00:00<00:00, 4661983.62it/s]




Model registered: iris-random-forest, version: 1


In [11]:
from azure.ai.ml.entities import ManagedOnlineEndpoint

endpoint = ManagedOnlineEndpoint(
    name="iris-endpoint-md2",
    description="Real-time endpoint for iris classifier",
    auth_mode="key"
)

ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Endpoint created successfully")

HttpResponseError: (BadRequest) The request is invalid.
Code: BadRequest
Message: The request is invalid.
Exception Details:	(InferencingClientCallFailed) {"error":{"code":"Validation","message":"{\"errors\":{\"\":[\"Specified endpoint [iris-endpoint-md2] has not been created successfully. Please recreate the endpoint.\"]},\"type\":\"https://tools.ietf.org/html/rfc9110#section-15.5.1\",\"title\":\"One or more validation errors occurred.\",\"status\":400,\"traceId\":\"00-51aeacf6b4cbbdd5f41304839090f2c9-0d6b084c1d38e75b-01\"}"}}
	Code: InferencingClientCallFailed
	Message: {"error":{"code":"Validation","message":"{\"errors\":{\"\":[\"Specified endpoint [iris-endpoint-md2] has not been created successfully. Please recreate the endpoint.\"]},\"type\":\"https://tools.ietf.org/html/rfc9110#section-15.5.1\",\"title\":\"One or more validation errors occurred.\",\"status\":400,\"traceId\":\"00-51aeacf6b4cbbdd5f41304839090f2c9-0d6b084c1d38e75b-01\"}"}}
Additional Information:Type: ComponentName